# Git and Running the Lesson Repo

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/07-git-and-repo.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your RAG pipeline is returning worse answers after last Friday's deploy. You ask the engineer who changed it what they did. "I just tweaked the prompt a bit," they say. You ask to see the diff. There is no diff: they edited the prompt string directly in the running Python file, tested it manually, said "that looks better," and deployed. The previous prompt is gone.

You now have no way to compare the current prompt against the one that was working. You cannot measure whether the change helped or hurt on your eval set. You cannot roll back. You are debugging a regression with no baseline.

This is not a hypothetical. It happens on every AI team that treats prompt text as configuration rather ...

## The Concept

### What Goes In Git vs What Stays Out

AI projects have a different risk profile from standard software projects. The wrong `.gitignore` can either leak secrets or lose reproducibility.

```
AI PROJECT DIRECTORY
+-----------------------+---------------------------+
|     GIT TRACKS        |    GITIGNORE EXCLUDES     |
+-----------------------+---------------------------+
| code/main.py          | .env (API keys!)           |
| prompts/              | __pycache__/               |
| evals/                | *.pyc                      |
| checks.json           | .venv/ (uv env)            |
| req...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 07: Git and Running the Lesson Repo
Phase 00: Setup and Mindset

Demonstrates the git workflow for AI projects:
  - Initializing a repo with the correct .gitignore
  - Committing a first Python script
  - Reviewing prompt changes with git diff before committing
  - Tagging working versions for rollback

Creates a demo repo at /tmp/ai_project_demo.
No API key required to run the git workflow parts.
"""

import os
import subprocess
import textwrap

### Shell helper

In [ ]:
def run(cmd: str, check: bool = True, cwd: str | None = None) -> subprocess.CompletedProcess:
    """
    Run a shell command, print it, and return the result.

    Args:
        cmd: Shell command to run.
        check: If True, raise on non-zero exit code.
        cwd: Working directory for the command.
    """
    print(f"  $ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True,
        cwd=cwd,
    )
    if result.stdout:
        print(textwrap.indent(result.stdout.rstrip(), "    "))
    if result.stderr and result.returncode != 0:
        print(textwrap.indent(f"stderr: {result.stderr.rstrip()}", "    "))
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

### .gitignore template for AI projects

In [ ]:
GITIGNORE_CONTENT = """\
# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
*.egg-info/
dist/
build/

# Virtual environments (uv, venv, conda)
.venv/
venv/
env/
.env/

# Secrets - NEVER commit these
.env
*.env
secrets.json
credentials.json
config.local.py
*_key.txt

# API response outputs that may contain PII
outputs/raw_responses/
outputs/eval_runs/
*.jsonl.gz

# Large files (model weights, binary data)
model_weights/
*.bin
*.pt
*.onnx
*.safetensors

# OS generated files
.DS_Store
.DS_Store?
Thumbs.db
ehthumbs.db

# IDE
.vscode/
.idea/
*.swp
*.swo

# Logs
*.log
logs/
"""

### First script template

In [ ]:
FIRST_SCRIPT_V1 = '''\
"""
First AI call - tracked in git so every prompt change is diffable.
Version 1: baseline prompt.
"""
import os
import anthropic

MODEL = "claude-3-5-haiku-20241022"

# Every change to this prompt is a behavioral change.
# Treat it like code: review the diff, commit with a meaningful message.
SYSTEM_PROMPT = """You are a concise assistant. Answer in 1-3 sentences.
Do not use preambles like \'Sure!\' or \'Great question!\'."""

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


def ask(question: str) -> str:
    """Single-turn question answering."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

### Demo

In [ ]:
q = "What is a context window in an LLM?"
print(f"Q: {q}")
print(f"A: {ask(q)}")

---

## Self-Check

Answer these without running code first:

**1. What is the primary problem with this situation from a debugging standpoint?**

- A. The commit message is too vague to know which specific change caused the regression.
- B. Git should not be used for prompt files - they should live in a database.
- C. The commit changed too many files - each change should have been a separate commit.
- D. Both A and C - the vague message and multi-file commit both make debugging harder.

**2. Which files should NOT be staged, and why?**

- A. Only .env - it contains a secret. The others are all safe to commit.
- B. .env (contains API key secret), __pycache__/ (auto-generated bytecode), and outputs/response_log.jsonl (may contain PII from user queries).
- C. Only __pycache__/ - cache files are too large for git.
- D. All files except main.py - only source code belongs in git.

**3. What git workflow step is missing before deploying the experiment branch?**

- A. Nothing is missing - if the score is higher, deploy immediately.
- B. The experiment branch should be merged to main and the merge commit tagged before deploying, so the deployed version is traceable in git history.
- C. The original v1.5-production tag should be deleted to avoid confusion.
- D. The experiment branch should be renamed to match the main branch name.

**4. Which commit message best follows AI project conventions?**

- A. 'update system prompt'
- B. 'fix yes/no responses'
- C. 'prompt: add yes/no prefix instruction - eval score for binary questions 60->90%'
- D. 'Changed line 14 of main.py to add yes/no instruction'

**5. When does the database approach for prompts create a specific debugging problem?**

- A. Never - databases have audit logs that are equivalent to git history.
- B. When you need to correlate a prompt change with a deployment timestamp, database audit logs and code git history are two separate systems that are hard to correlate when debugging regressions.
- C. Only for small teams - large teams always use databases for config.
- D. When prompts are longer than 1,000 tokens - databases cannot store strings that long.

**6. What is the correct git workflow for this experiment?**

- A. Edit the prompt on main, run evals, then revert if it does not work.
- B. Create a branch (e.g., experiment/chain-of-thought), make the changes there, run evals on the branch, and only merge to main if the score improves.
- C. Create a copy of main.py named main_cot.py, edit it, and delete it when done.
- D. Use git stash to save the current prompt, try the experiment, then stash pop if it fails.

**7. Which git commands give you the fastest path to understanding the v1.3 regression?**

- A. git status - shows the current state and which files changed at v1.3.
- B. git show v1.3 to see the exact changes in that tag, then git diff v1.2 v1.3 to compare the two versions.
- C. git log --all to see every commit, then manually read through the history.
- D. git blame to find who made each change in the current version of the file.

**8. What is the specific advantage of git tags over commit hashes for AI project versioning?**

- A. Tags are technically required for git to track file history - hashes alone are not enough.
- B. Tags are human-readable names attached to specific commits, so 'v2.0-chain-of-thought' is immediately meaningful while a hash like 'a7f3c91' is not - especially when correlating with eval scores and deploy dates.
- C. Tags reduce the size of the git repository by compressing commit history.
- D. Tags are required for CI/CD systems to deploy code - hashes cannot trigger deployments.

_Answers are in `checks.json` in the lesson directory._